In [ ]:
import pandas as pd
import numpy as np

loans = pd.read_csv('data/library_loans_clean.csv', parse_dates=['loan_date', 'return_date'])
members_raw = pd.read_csv('data/library_members.csv')
members = members_raw.copy()

## Lab 4 - Part 1
### Ex 1

In [ ]:
print(loans.shape, loans.dtypes)

In [ ]:
print(loans.isna().sum(),
     (loans.isna().mean() * 100).round(1))
loans.info()

#### loan_date has 3, 0.5% missing values and return_date has 36, 6%
### Ex 2

In [ ]:
missing_loan_date = loans['loan_date'].isna()

loans[missing_loan_date]

In [ ]:
loans[loans['return_date'].isna()]

#### All missing loan_date have a return_date and returned=True so they are real loans missing the data.
#### All missing return_date have a loan_date but they are mixed returned=True/False. The ones with returned=True would be missing data.

In [ ]:
loans['has_return_date'] = loans['return_date'].notna()

loans['has_return_date'].sum()

In [ ]:
returned_missing_return_date = (loans['returned'] & ~loans['has_return_date'])

loans[returned_missing_return_date]

#### 6 total loans fall in the faulty data category with no return_date and returned=True
### Ex 3

In [ ]:
print(members.shape)
print(members.info())
members.describe()

In [ ]:
wonky_birth_year = members['birth_year'].isin([1900,2030])
duplicate_member_both = members.duplicated(keep=False)
duplicate_member = members.duplicated()

print(len(members[wonky_birth_year]))
print(members[duplicate_member_both])
members[wonky_birth_year]

#### Shape: 148 rows, 3 columns. 2. 
#### 18 missing birth_year, 12 missing branch
#### Bad birth_year values: 1900, 2030. 11 entries use these.
#### 4 duplicates each pair adjacent rows and same member_id. true duplicates.
### Ex 4

In [ ]:
members['birth_year'] = members['birth_year'].where(members['birth_year'].between(1920, 2015))

In [ ]:
members.describe()

#### The 11 faulty birth_year values are now NaN and the describe look a lot better.
### Ex 5 a)

In [ ]:
members.dropna(subset=['branch'])

In [ ]:
members.dropna(subset=['birth_year'])

#### dropna results: branch: 107, birth_year: 119
### Ex 5 b)

In [ ]:
members['branch'].value_counts()

In [ ]:
members['branch'] = members['branch'].where(members['branch'].notna(), other='Unknown')

In [ ]:
print(members.info())
members['branch'].value_counts()

##### Now the null branch values are replaced with placeholder 'Unknown'. 
##### Central    40
##### South      34
##### North      32
##### East       17
##### Harbour    13
##### Unknown    12
### Ex 5 c)

#### We keep missing birth year as null. Guessing will only lessen the data quality not improve it.
### Ex 6

In [ ]:
print(loans.shape)
print(members.shape)
print(loans['late_fee'].sum())

In [ ]:
merged = loans.merge(members, how='left')

print(merged['late_fee'].sum())
merged.shape

In [ ]:
# merged_validate = loans.merge(members, how='left', validate='many_to_one')
# MergeError: Merge keys are not unique in right dataset; not a many-to-one merge

#### pre-merge: (600, 9) (148, 3) 7487.5. post-merge: (616, 11) 7725.0. 616 because the 4 duplicate members are still there.
### Ex 7

In [ ]:
members = members[~duplicate_member]

print(loans.shape)
print(members.shape)

In [ ]:
real_merged = loans.merge(members, how='left', indicator=True, validate='many_to_one')
left_only = real_merged['_merge'] == 'left_only'

print(real_merged['late_fee'].sum())
print(real_merged.shape)
(real_merged.loc[left_only]
    .agg(num=('loan_id','count'),memb=('member_id','nunique'),late_fee=('late_fee','sum')))

In [ ]:
inner_merged = loans.merge(members, how='inner', indicator=True, validate='many_to_one')

print(inner_merged['late_fee'].sum())
print(inner_merged.shape)

#### left_only loans: 31, members: 6, fee total: 347.5
#### inner result: (569, 12) 7140.0
### Ex 8

In [ ]:
real_merged['branch'] = real_merged['branch'].fillna('No member record')

real_merged['branch'].value_counts()

real_merged.to_csv('data/merged.csv', index=False)

#### Central 144, South 130, North 126, East 72, Harbour 51, Unknown 46, No member record 31
#### Unknown is a member without a set branch. No member record are loans without a member.